In [ ]:
# Copyright (c) TorchGeo Contributors. All rights reserved.
# Licensed under the MIT License.

# Spatiotemporal Segmentation with ConvLSTM
_Written by: Yi-Chia Chang_

In this tutorial, we train a [Convolutional LSTM](https://arxiv.org/abs/1506.04214) from scratch for time-series semantic segmentation, then run cross-region inference on raw Sentinel-2 imagery and evaluate against an independent ground-truth dataset. ConvLSTM replaces the dense matrix multiplications inside a regular LSTM cell with 2D convolutions, so each gate operates on a `[C, H, W]` feature map instead of a flat vector — this preserves spatial structure while still modelling temporal dynamics. Crop-type mapping is a natural fit: different crops have distinctive phenological signatures that unfold across the growing season.

The full pipeline is just three components glued together by torchgeo:

1. **Train** on [PASTIS100](https://torchgeo.readthedocs.io/en/stable/api/datasets/pastis.html) (French agricultural parcels, Sentinel-2 time series) using `SpatioTemporalSegmentationTask` from [PR #3671](https://github.com/torchgeo/torchgeo/pull/3671).
2. **Infer** over central North Rhine–Westphalia, Germany, by composing the new `GridSpatialSampler @ SequentialPeriodSampler` from [PR #3552](https://github.com/torchgeo/torchgeo/pull/3552) over a `Sentinel2(time_series=True)` raster dataset.
3. **Evaluate** against [EuroCrops](https://www.eurocrops.tum.de/) harmonized crop labels.

## Setup

In [ ]:
%pip install torchgeo tensorboard pystac-client planetary-computer odc-stac

## Imports

In [ ]:
%matplotlib inline

import os
import tempfile
from pathlib import Path

import kornia.augmentation as K
import lightning as L
import matplotlib.pyplot as plt
import numpy as np
import torch
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from rasterio.features import rasterize
from rasterio.transform import from_bounds
from tqdm.auto import tqdm

from torchgeo.datamodules import PASTIS100DataModule
from torchgeo.datasets import PASTIS100, EuroCrops, Sentinel2
from torchgeo.samplers import GridSpatialSampler, SequentialPeriodSampler
from torchgeo.trainers import SpatioTemporalSegmentationTask

torch.set_float32_matmul_precision('medium')
L.seed_everything(0, workers=True)

## Visualize the dataset

PASTIS100 ships 100 patches at 128×128 with a variable-length Sentinel-2 time series per patch and a static 20-class semantic mask (background + 18 crop classes + a *void* label for parcels mostly outside the patch). We use all 10 L2A bands.

In [ ]:
root = os.path.join(tempfile.gettempdir(), 'pastis100')
dataset = PASTIS100(root=root, bands=PASTIS100.s2_bands, download=True)

fig = dataset.plot(dataset[2], suptitle='PASTIS100 sample')
plt.show()
print(f'{len(dataset)} patches, image {tuple(dataset[2]["image"].shape)} (T, C, H, W)')

## Train

`PASTIS100DataModule` builds the train/val/test split and pads each time series to a common length, and `SpatioTemporalSegmentationTask` (PR #3671) wires the `ConvLSTM` model up to the standard loss / metric / predict steps. The only customization we need is overriding the datamodule's default per-batch augmentation to divide Sentinel-2 reflectance by 10000 (the L2A scaling factor) rather than the default 255.

In [ ]:
datamodule = PASTIS100DataModule(
    root=root, bands=PASTIS100.s2_bands, batch_size=4, num_workers=4, padding_length=30,
)
datamodule.aug = K.AugmentationSequential(
    K.VideoSequential(K.Normalize(mean=torch.tensor(0.0), std=torch.tensor(10000.0))),
    data_keys=None, keepdim=True,
)
task = SpatioTemporalSegmentationTask(
    model='convlstm', in_channels=10, num_classes=20, ignore_index=19,
    hidden_dim=64, num_layers=2, kernel_size=3,
)
trainer = L.Trainer(
    max_epochs=30, accelerator='auto', devices=1,
    callbacks=[
        EarlyStopping('val_loss', patience=5),
        ModelCheckpoint(monitor='val_loss', mode='min'),
    ],
)
trainer.fit(task, datamodule=datamodule)

## Inference on raw Sentinel-2 over Germany

The spatiotemporal samplers compose a spatial chip sampler with a temporal slicer over any `RasterDataset(time_series=True)`. The canonical pattern for cross-region inference is:

```python
spatial = GridSpatialSampler(s2, size=128, stride=128)
temporal = SequentialPeriodSampler(s2, freq='Y')
sampler = spatial @ temporal  # one chip × one yearly time-series slice
```

We first pull a year of cloud-filtered L2A imagery over central NRW from Microsoft Planetary Computer, save it in a torchgeo-compatible naming pattern, then wrap it as a `Sentinel2` `RasterDataset`.

In [ ]:
import odc.stac
import pandas as pd
import planetary_computer
import pystac_client

S2_DIR = Path(tempfile.gettempdir()) / 's2_germany'
S2_DIR.mkdir(parents=True, exist_ok=True)
BBOX = (7.0, 51.3, 7.5, 51.6)  # central NRW, lat/lon
CRS = 'EPSG:32632'
BANDS = ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']

catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace,
)
items = sorted(
    catalog.search(
        collections=['sentinel-2-l2a'],
        bbox=BBOX,
        datetime='2019-01-01/2019-12-31',
        query={'eo:cloud_cover': {'lt': 50}},
    ).get_items(),
    key=lambda x: x.datetime,
)
stack = odc.stac.load(
    items, bands=BANDS, bbox=BBOX, resolution=10, crs=CRS,
    chunks={'time': 1, 'x': 1024, 'y': 1024}, groupby='solar_day',
)
for t in tqdm(range(stack.sizes['time']), desc='Save GeoTIFFs'):
    date = pd.to_datetime(stack.time.values[t]).strftime('%Y%m%dT%H%M%S')
    for band in BANDS:
        out = S2_DIR / f'TILE_{date}_{band}.tif'
        if not out.exists():
            stack[band].isel(time=t).rio.to_raster(out, dtype='uint16', compress='DEFLATE')

Wrap the downloaded GeoTIFFs as a `Sentinel2` dataset with `time_series=True`, compose the new samplers, and run inference patch by patch into a single mosaic.

In [ ]:
class GermanyS2(Sentinel2):
    filename_glob = 'TILE_*_B*.tif'
    filename_regex = r'TILE_(?P<date>\d{8}T\d{6})_(?P<band>B\d{2}A?)\.tif'
    date_format = '%Y%m%dT%H%M%S'
    separate_files = True
    all_bands = BANDS

s2 = GermanyS2(paths=str(S2_DIR), bands=BANDS, time_series=True)
sampler = GridSpatialSampler(s2, size=128, stride=128) @ SequentialPeriodSampler(s2, freq='Y')

H = int(round((s2.bounds.maxy - s2.bounds.miny) / s2.res[1]))
W = int(round((s2.bounds.maxx - s2.bounds.minx) / s2.res[0]))
transform = from_bounds(s2.bounds.minx, s2.bounds.miny, s2.bounds.maxx, s2.bounds.maxy, W, H)
prediction = np.zeros((H, W), dtype=np.uint8)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
task = task.to(device).eval()
with torch.no_grad():
    for sl in tqdm(sampler, desc='Inference'):
        img = s2[sl]['image'].float() / 10000.0
        if img.ndim == 3:  # single-timestep chip
            img = img.unsqueeze(0)
        logits = task(img.unsqueeze(0).to(device))
        pred = logits.argmax(dim=1).squeeze(0).cpu().numpy().astype(np.uint8)
        col = int(round((sl[0].start - transform.c) / s2.res[0]))
        row = int(round((transform.f - sl[1].stop) / s2.res[1]))
        h, w = pred.shape
        prediction[row:row + h, col:col + w] = pred

## Evaluation against EuroCrops

[EuroCrops](https://www.eurocrops.tum.de/) harmonizes crop parcel polygons across the EU using the HCAT3 taxonomy. We download the German layer, map HCAT3 codes to PASTIS class IDs via prefix matching, rasterize onto the same 10 m grid as our predictions, and compute per-class IoU on the overlapping pixels.

The mapping below is approximate — cross-check against the HCAT3 taxonomy CSV in your EuroCrops download for research-grade use. Several PASTIS classes (mixed cereal, fruits/vegetables/flowers) have no clean 1-to-1 correspondence in HCAT3.

In [ ]:
import geopandas as gpd

EuroCrops(paths=str(Path(tempfile.gettempdir()) / 'eurocrops'), download=True)
shp = next(Path(tempfile.gettempdir()) / 'eurocrops'.rglob('*DE*_EC*.shp'))
gdf = gpd.read_file(shp).to_crs(CRS).cx[
    s2.bounds.minx:s2.bounds.maxx, s2.bounds.miny:s2.bounds.maxy
]

# HCAT3 prefix → PASTIS class ID (verify against your HCAT3 taxonomy CSV)
HCAT_TO_PASTIS = {
    '3301060': 1, '3301010100': 2, '3301070': 3, '3301020100': 4, '3304010100': 5,
    '3301020200': 6, '3304020': 7, '3306': 8, '3304060': 9, '3301050100': 10,
    '3301010300': 11, '3305': 12, '3304070': 13, '3303': 14, '3303060': 15,
    '3306020': 16, '3301080': 17, '3301040': 18,
}

def hcat_to_pastis(code: object) -> int:
    if not isinstance(code, str):
        return 0
    for prefix, cls in HCAT_TO_PASTIS.items():
        if code.startswith(prefix):
            return cls
    return 0

gdf['pastis_class'] = gdf['EC_hcat_c'].map(hcat_to_pastis)
ground_truth = rasterize(
    [(g, c) for g, c in zip(gdf.geometry, gdf['pastis_class']) if c > 0],
    out_shape=(H, W), transform=transform, fill=0, dtype='uint8',
)

valid = (ground_truth > 0) & (prediction != 19)
y_true, y_pred = ground_truth[valid], prediction[valid]
ious = []
for c in range(1, 19):
    inter = ((y_true == c) & (y_pred == c)).sum()
    union = ((y_true == c) | (y_pred == c)).sum()
    if union > 0:
        ious.append(inter / union)
print(f'Overall accuracy: {(y_true == y_pred).mean():.3f}')
print(f'Mean IoU:         {np.mean(ious):.3f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(prediction, cmap='tab20', vmin=0, vmax=19)
axes[0].set_title('ConvLSTM predictions (PASTIS → NRW)')
axes[1].imshow(ground_truth, cmap='tab20', vmin=0, vmax=19)
axes[1].set_title('EuroCrops ground truth')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()